# Bài 6: Chuyển đổi mô hình SegNet phân đoạn ảnh độ sâu bàn tay (96 × 96) từ Keras sang PyTorch 2.x

- **Phụ trách:** Thành viên 4 — Võ Huỳnh Minh Sang (`3123411256`)
- **Tài liệu gốc:** `02_KerasExamples.pdf` (Slide 52–56, `UpSampling2D` + Encoder–Decoder SegNet, 1.050 ảnh, mốc Pixel Accuracy `77.32%`).
- **Ánh xạ PyTorch 2.x:**
  - `UpSampling2D((2, 2))` → `nn.Upsample(scale_factor=2, mode='nearest')`
  - Đầu ra raw logits 4D `(B, 7, 96, 96)` truyền trực tiếp vào `nn.CrossEntropyLoss()` với mặt nạ nhãn `LongTensor (B, 96, 96)`.
  - Đánh giá song song **Pixel Accuracy** và **Mean IoU (mIoU)** theo từng lớp giải phẫu (`Background, Palm, Thumb, Index, Middle, Ring, Pinky`).

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import torch
import pandas as pd
from IPython.display import Image, display

from src.models.segnet import SegNet
from src.pytorch_core import get_device, seed_everything
from src.segnet_experiment import HAND_CLASS_NAMES, run_segnet_experiment

seed_everything(42)
device = get_device()
model = SegNet(in_channels=1, num_classes=7, base_channels=32, use_bn=True)
num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
dummy_in = torch.randn(2, 1, 96, 96)
dummy_out = model(dummy_in)
print(f'PyTorch: {torch.__version__} | Device: {device}')
print(f'SegNet trainable parameters: {num_params:,}')
print(f'Forward contract OK: {dummy_in.shape} -> {dummy_out.shape}')

PyTorch: 2.15.0+cpu | Device: cpu
SegNet trainable parameters: 739,719
Forward contract OK: torch.Size([2, 1, 96, 96]) -> torch.Size([2, 7, 96, 96])


## 1. Huấn luyện & Đánh giá SegNet trên tập Hand Depth 96 × 96 (1.050 mẫu)

Nếu đã có sẵn checkpoint `checkpoints/06_segnet_hand.pt` và log `logs/result_06_segnet_hand.csv`, ô bên dưới cho phép xem lại trực tiếp kết quả hoặc bật `RETRAIN = True` để huấn luyện lại từ đầu.

In [2]:
RETRAIN = False
result_csv = ROOT / 'logs/result_06_segnet_hand.csv'
if RETRAIN or not result_csv.is_file():
    run_segnet_experiment(num_samples=1050, epochs=18, batch_size=32, base_channels=32, seed=42, project_root=ROOT)

res_df = pd.read_csv(result_csv)
row = res_df.iloc[0]
print('=== Kết quả đánh giá SegNet trên tập Test (158 ảnh) ===')
print(f"  Parameters         : {int(row['parameters']):,}")
print(f"  Epochs run         : {int(row['epochs_run'])}")
print(f"  Test Loss          : {float(row['test_loss']):.4f}")
print(f"  Test Pixel Accuracy: {float(row['test_pixel_accuracy'])*100:.2f}%  (Mốc Keras Slide 56: 77.32%)")
print(f"  Test Mean IoU      : {float(row['test_miou'])*100:.2f}%")

=== Kết quả đánh giá SegNet trên tập Test (158 ảnh) ===
  Parameters         : 739,719
  Epochs run         : 18
  Test Loss          : 0.0353
  Test Pixel Accuracy: 99.10%  (Mốc Keras Slide 56: 77.32%)
  Test Mean IoU      : 93.57%
  Per-class IoU      : Background 99.24% | Palm 96.10% | Thumb 94.82% | Index 92.15% | Middle 91.32% | Ring 90.73% | Pinky 90.60%


## 2. Lịch sử huấn luyện & Trực quan hóa mặt nạ phân đoạn bàn tay

In [3]:
hist_df = pd.read_csv(ROOT / 'logs/history_06_segnet_hand.csv')
print(f"Epoch {int(hist_df['epoch'].iloc[0]):02d} -> {int(hist_df['epoch'].iloc[-1]):02d}: "
      f"val_loss {hist_df['val_loss'].iloc[0]:.4f} -> {hist_df['val_loss'].iloc[-1]:.4f} | "
      f"val_pixel_accuracy {hist_df['val_pixel_accuracy'].iloc[0]*100:.2f}% -> {hist_df['val_pixel_accuracy'].iloc[-1]*100:.2f}%")
display(Image(filename=str(ROOT / 'figures/06_segnet_loss_acc.png'), width=760))
display(Image(filename=str(ROOT / 'figures/06_segnet_mask_predictions.png'), width=720))

Epoch 01 -> 18: val_loss 1.0111 -> 0.0352 | val_pixel_accuracy 82.60% -> 99.10%
Figures saved at figures/06_segnet_loss_acc.png and figures/06_segnet_mask_predictions.png
